# arange-fancy-index-cross-entropy — worked example 1: Gather per-sample log-probabilities at the true class

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `arange-fancy-index-cross-entropy`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

The `arange(B)` fancy-index idiom pairs a row index `[0,1,...,B-1]` with a `(B,)` label tensor positionally, yielding one element per row. Here we apply it to `log_softmax` output to read off the correct-class log-probability for each sample. This is exactly the NLL gather step.

## Worked solution

**Goal.** Given `logits` of shape `(B, C)` and integer `target` of shape `(B,)`, return the `(B,)` vector whose entry `i` is `log_softmax(logits)[i, target[i]]`.

**Step 1 — log-softmax.** Compute `logp = t.log_softmax(logits, dim=-1)`, shape `(B, C)`. This normalizes each row so the row-wise `exp` sums to 1, then takes the log. Doing log-softmax in one call is numerically stable (it subtracts the row max internally).

**Step 2 — build the row index.** `t.arange(B)` is `[0, 1, ..., B-1]`, the same length as `target`. When two 1-D index tensors of equal length index a 2-D tensor, PyTorch zips them into pairs `(0, target[0]), (1, target[1]), ...`. That is why `arange` advances the row axis in lockstep with the column axis.

**Step 3 — gather.** `logp[t.arange(B), target]` produces the `(B,)` vector of correct-class log-probabilities. No loop, one element per row.

**Why not `logp[:, target]`?** That would broadcast to `(B, B)` — every row indexed by every label — which is not what we want. The `arange` is what keeps the pairing diagonal.

In [ ]:
def pick_target_logprobs(logits, target):
    B = logits.shape[0]
    logp = t.log_softmax(logits, dim=-1)          # (B, C)
    return logp[t.arange(B), target]              # (B,)

t.manual_seed(0)
logits = t.randn(4, 5)
target = t.tensor([2, 0, 4, 1])
out = pick_target_logprobs(logits, target)
print(out.shape)
print(out)